In [8]:
import os
os.chdir("..")  # 상위 폴더(ai)로 이동

gam

In [23]:
import datetime
import pandas as pd
import joblib

from utils.naver_searchad_relkeyword import fetch_relkwdstat
from utils.naver_shoppinginsite_search import fetch_category_keyword_data

# ===== 상수 선언 =====
KEYWORD = "참깨라면"
BASE_PRICE = 5000
PRICE_FACTORS = [0.6, 0.8, 1.0, 1.2, 1.4]
PUM_NAME = "라면"
CATEGORY_CODE = "50000006"

# ===== 모델 불러오기 =====
gam_package = joblib.load("./models/gam_model.pkl")
gam = gam_package["model"]
X_columns = gam_package["columns"]

# ===== 최근 4주 클릭 데이터 =====
rel_data = fetch_relkwdstat([KEYWORD])
recent_4weeks_click_avg = rel_data[0].get("최근4주클릭수평균", 0) if rel_data else 0

today = pd.Timestamp.today()
start_date = today - pd.DateOffset(days=30)
end_date = today

try:
    search_df = fetch_category_keyword_data(
        start_date=start_date.strftime("%Y-%m-%d"),
        end_date=end_date.strftime("%Y-%m-%d"),
        category=CATEGORY_CODE,
        keyword=KEYWORD
    )
except Exception as e:
    print(f"⚠️ API 호출 실패: {e}")
    search_df = pd.DataFrame(columns=["날짜","클릭량"])

prev_sum = search_df["클릭량"].sum() if not search_df.empty else 0
recent_4weeks_click_total = recent_4weeks_click_avg * 30
recent_ratio = (recent_4weeks_click_total / prev_sum) if prev_sum > 0 else 0

# ===== 예측 데이터 생성 및 수행 =====
rows = []
dates = []

for day_offset in range(0, 7):  # 오늘 포함 7일
    target_date = datetime.date.today() + datetime.timedelta(days=day_offset)
    weekday = target_date.weekday()
    dates.append(target_date.strftime("%Y-%m-%d"))

    row = []
    for factor in PRICE_FACTORS:
        price = int(BASE_PRICE * factor)

        test_data = {
            "1개당가격": price,
            "최근4주클릭수_비율": recent_ratio,
            "weekday": weekday,
            "pum_name": PUM_NAME
        }
        test_df = pd.DataFrame([test_data])
        test_df = pd.get_dummies(test_df, columns=["pum_name","weekday"], drop_first=True)
        test_df = test_df.reindex(columns=X_columns, fill_value=0)

        pred_clicks = gam.predict(test_df)[0]
        row.append(pred_clicks)
    rows.append(row)

# ===== 결과를 DataFrame으로 정리 =====
result_df = pd.DataFrame(rows, index=dates, columns=[f"{int(f*100)}%" for f in PRICE_FACTORS])
print(result_df)

                  60%        80%       100%       120%      140%
2026-04-10  58.931278  50.299962  47.345822  44.875067  43.31315
2026-04-11  58.931278  50.299962  47.345822  44.875067  43.31315
2026-04-12  58.931278  50.299962  47.345822  44.875067  43.31315
2026-04-13  58.931278  50.299962  47.345822  44.875067  43.31315
2026-04-14  58.931278  50.299962  47.345822  44.875067  43.31315
2026-04-15  58.931278  50.299962  47.345822  44.875067  43.31315
2026-04-16  58.931278  50.299962  47.345822  44.875067  43.31315


mlp

In [ ]:
import datetime
import pandas as pd
import joblib
import torch
import torch.nn as nn

from utils.naver_searchad_relkeyword import fetch_relkwdstat
from utils.naver_shoppinginsite_search import fetch_category_keyword_data

# ===== 상수 선언 =====
KEYWORD = "참깨라면"
BASE_PRICE = 1000
PRICE_FACTORS = [0.6, 0.8, 1.0, 1.2, 1.4]
PUM_NAME = "라면"
CATEGORY_CODE = "50000006"

# ===== 모델 불러오기 =====
nn_package = joblib.load("./models/nn_model.pkl")
input_dim = nn_package["input_dim"]
columns = nn_package["columns"]
scaler = nn_package["scaler"]

class SimpleMLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super(SimpleMLPRegressor, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleMLPRegressor(input_dim=input_dim)
model.load_state_dict(nn_package["model_state"])
model.eval()

# ===== 최근 4주 클릭 데이터 =====
rel_data = fetch_relkwdstat([KEYWORD])
recent_4weeks_click_avg = rel_data[0].get("최근4주클릭수평균", 0) if rel_data else 0

today = pd.Timestamp.today()
start_date = today - pd.DateOffset(days=30)
end_date = today

try:
    search_df = fetch_category_keyword_data(
        start_date=start_date.strftime("%Y-%m-%d"),
        end_date=end_date.strftime("%Y-%m-%d"),
        category=CATEGORY_CODE,
        keyword=KEYWORD
    )
except Exception as e:
    print(f"⚠️ API 호출 실패: {e}")
    search_df = pd.DataFrame(columns=["날짜","클릭량"])

prev_sum = search_df["클릭량"].sum() if not search_df.empty else 0
recent_4weeks_click_total = recent_4weeks_click_avg * 30
recent_ratio = (recent_4weeks_click_total / prev_sum) if prev_sum > 0 else 0

# ===== 예측 수행 =====
rows = []
dates = []

for day_offset in range(0, 7):
    target_date = datetime.date.today() + datetime.timedelta(days=day_offset)
    weekday = target_date.weekday()
    dates.append(target_date.strftime("%Y-%m-%d"))

    row = []
    for factor in PRICE_FACTORS:
        price = int(BASE_PRICE * factor)

        test_data = {
            "1개당가격": price,
            "최근4주클릭수_비율": recent_ratio,
            "weekday": weekday,
            "pum_name": PUM_NAME
        }
        test_df = pd.DataFrame([test_data])
        test_df = pd.get_dummies(test_df, columns=["pum_name","weekday"], drop_first=True)
        test_df = test_df.reindex(columns=columns, fill_value=0)

        # 스케일링
        X_scaled = scaler.transform(test_df)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

        with torch.no_grad():
            pred_clicks = model(X_tensor).cpu().numpy()[0][0]

        row.append(pred_clicks)
    rows.append(row)

# ===== 결과를 DataFrame으로 정리 =====
result_df = pd.DataFrame(rows, index=dates, columns=[f"{int(f*100)}%" for f in PRICE_FACTORS])
print(result_df)

                  60%        80%       100%       120%       140%
2026-04-11  46.188564  44.837421  43.486225  42.135014  40.783863
2026-04-12  46.188564  44.837421  43.486225  42.135014  40.783863
2026-04-13  46.188564  44.837421  43.486225  42.135014  40.783863
2026-04-14  46.188564  44.837421  43.486225  42.135014  40.783863
2026-04-15  46.188564  44.837421  43.486225  42.135014  40.783863
2026-04-16  46.188564  44.837421  43.486225  42.135014  40.783863
2026-04-17  46.188564  44.837421  43.486225  42.135014  40.783863
